In [0]:
%sql 
USE flights_silver;

### Implementacja SCD2

Używamy logiki SCD2, aby zachować historię zmian atrybutów. Celowo powielamy wiersze, aby umożliwić połączenie historycznych faktów z poprawnmi danymi (np. nazwą linii), które wtedy obowiązywały. 

Do każego rekordu dopisujemy 3 kolumny: *is_current*, *start_time*, *end_time*, któe będą wskazywać na to, czy informacje są dalej aktualne, a jeśli nie są, to od kiedy do kiedy aktualne były. Nie znamy historycznej daty wejścia konkretnych rekordów do użycia, więc dla pierwszego załadaowania używamy "dummy date": 1900-01-01 (uzupełnianie *start_time* aktualną datą nie ma sensu, jako że dane dotyczą lotów z roku 2015). 

Dzięki temu podejściu, jeśli któraś z linii lotniczych zmieni nazwę, rekord zostanie zamknięty (*is_current* zostanie ustawione na False, a *end_time* przestanie byc nullem, a datą, do której poprzednia nazwa funkcjonowała). Dla danej linii do tabeli zostanie dopisany nowy wiersz, zawierający aktualne informacje.

In [0]:
%python
from pyspark.sql.functions import lit, col, to_date

df_airlines_raw = spark.table("flights_bronze.airlines")

df_dim_airlines = df_airlines_raw.select(
    col("iata_code").alias("airline_key"), # klucz identyfikujący (do odseparowania przyszłych zmian)
    col("iata_code"), # oryginalna kolumna
    col("airline").alias("airline_name"), # alias dla przejrzystości

    lit(True).alias("is_current"),
    to_date(lit("1900-01-01")).alias("start_date"), # dummy data, bo nie wiemy od kiedy ważne
    to_date(lit(None)).alias("end_date") # wciąż jest aktywne => end_date = null
) 

df_dim_airlines.write.mode("overwrite").insertInto("flights_silver.dim_airlines")


In [0]:
%python
display(df_dim_airlines.limit(5))

airline_key,iata_code,airline_name,is_current,start_date,end_date
UA,UA,United Air Lines Inc.,true,1900-01-01,null
AA,AA,American Airlines Inc.,true,1900-01-01,null
US,US,US Airways Inc.,true,1900-01-01,null
F9,F9,Frontier Airlines Inc.,true,1900-01-01,null
B6,B6,JetBlue Airways,true,1900-01-01,null


### Implementacja procesu ładowania przyrostowego

Nowe dane łączymy jedynie z aktualnymi rekordami, nie biorąc pod uwagę tych historycznych. 

Jeśli zgadza się *iata_code* i *is_current* ustawione jest na True, możemy działać dalej. Jeśli któraś z wartości się zmieniła (tutaj jedynie nazwa linii lotniczej), musimy zamknąć rekord -> *is_current* ustawić na False, a za *end_date* wstawić aktualną datę (datę, kiedy ładujemy nowe dane). 

Na koniec ładujemy nowe lub zmienione rekordy (*is_current*=True, *start_date*=dzisiaj, *end_date*=null)

In [0]:
%python
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit, current_date

airlines_silver = "flights_silver.dim_airlines" 
airlines_bronze = spark.table("flights_bronze.airlines") # aktualny stan danych w warstwie bronze

target_table = DeltaTable.forName(spark, airlines_silver)

(target_table.alias("target")
 .merge(
   airlines_bronze.alias("source"),
   # łaczymy tylko, jeśli zgadza się klucz biznesowy (iata code) i jeśli rekord jest aktywny
   "target.iata_code = source.iata_code AND target.is_current = true"
)
 # rekord istnieje, ale jakaś wartość się zmieniła -> zamykamy stary rekord
 .whenMatchedUpdate(
     condition=(
         col("target.airline_name") != col("source.airline")
     ),
     set={
         "is_current": lit(False), # już nieaktualne
         "end_date": current_date() # obecna data datą zamknięcia
     }
 )
 # rekord jest nowy lub został zamknięty -> tworzymy nowy
 .whenNotMatchedInsert(
     values={
         "airline_key": col("source.iata_code"),
         "iata_code": col("source.iata_code"),
         "airline_name": col("source.airline"),
         "is_current": lit(True),
         "start_date": current_date(),
         "end_date": lit(None).cast("date")
     }
 )
 .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]